# Sparkov Fraud Detection: M3 Pseudo-Labelling with LightGBM

## Objective

This notebook evaluates **M3 confidence-based pseudo-labelling** as an extension of the Sparkov **M2 adaptive LightGBM** configuration. It retains expanding historical training data, retraining every seven days, a seven-day label-verification delay, and class balancing based on verified labels.

At each retraining cutoff, a model fitted on verified observations scores transactions whose labels are not yet available. M3 selects pseudo-labels by **rank**, rather than applying fixed fraud-probability cutoffs. This follows the pseudo-labelling selection and class-specific reliability-weighting design used in the IEEE-CIS M3 notebook, while retaining Sparkov's selected LightGBM configuration and class-balancing approach.

### Four tuned M3 parameters

- **`fraud_thres`:** select the highest-scoring fraction of the unverified pool, with the target fraction equal to `fraud_thres × verified fraud rate`. Assign these transactions pseudo-fraud labels (1).
- **`legit_thres`:** select the lowest-scoring fraction of the unverified pool, equal to `legit_thres`. Assign these transactions pseudo-legitimate labels (0), except where a transaction also qualifies as pseudo-fraud; fraud takes precedence.
- **`fraud_reliability`:** reliability multiplier applied to selected pseudo-fraud examples.
- **`legit_reliability`:** reliability multiplier applied to selected pseudo-legitimate examples.

Selection counts are rounded and clipped to at least one candidate when the pool is nonempty. Score cutoffs are inclusive, so ties can produce more selected transactions than the nominal counts. Unselected transactions are excluded from the refit.

### Training weights

Balanced class weights are calculated **from verified labels only** at each retraining cutoff. The LightGBM fit uses per-row `sample_weight` rather than also enabling estimator-level class weighting:

`sample weight = verified-derived class weight × reliability multiplier`

| Training observation | Class-weight component | Reliability multiplier |
| --- | --- | --- |
| Verified legitimate | Verified-derived weight for class 0 | 1.0 |
| Verified fraud | Verified-derived weight for class 1 | 1.0 |
| Pseudo-legitimate | Verified-derived weight for class 0 | `legit_reliability` |
| Pseudo-fraud | Verified-derived weight for class 1 | `fraud_reliability` |

This preserves Sparkov M2's class-balancing strategy while discounting uncertain pseudo-labels. It differs from an unweighted verified-sample baseline, so cross-dataset comparisons should note the dataset-specific base-model weighting.

### Development grid and evaluation

A compact, prespecified **16-configuration grid** replaces IEEE-CIS's larger Optuna search:

```python
FRAUD_THRESHOLDS = [0.15, 0.50]
LEGIT_THRESHOLDS = [0.25, 0.50]
FRAUD_RELIABILITIES = [0.15, 0.50]
LEGIT_RELIABILITIES = [0.30, 0.75]
```

The final **84 days of the training partition** are used to select the M3 configuration by development PR-AUC. The selected configuration is then evaluated on the held-out **validation partition** against a verified-only M2 LightGBM baseline under the same temporal setup. **The test partition is not loaded or used in this notebook.**


In [1]:
from pathlib import Path
from itertools import product
import time
import json
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/sparkov/m3_rank_aligned_ieee")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load the processed feature table and the pre-approved modelling feature list.
df = pd.read_pickle(DATA_DIR / "sparkov_features.pkl")
with open(DATA_DIR / "feature_lists.json", "r") as f:
    feature_lists = json.load(f)
feature_cols = feature_lists["sparkov_feature_cols"]

TIME_COL = "trans_date_trans_time"
TARGET_COL = "is_fraud"
SPLIT_COL = "sparkov_split"

# The test partition is excluded. Retain chronological train and validation only.
df[TIME_COL] = pd.to_datetime(df[TIME_COL])
m3_df = df.loc[df[SPLIT_COL].isin(["train", "validation"])].copy()
m3_df = m3_df.sort_values(TIME_COL, kind="stable").reset_index(drop=True)
del df

# Match M2: reconstruct selected frequency features using strictly earlier rows.
frequency_sources = {
    "merchant_freq": "merchant",
    "category_freq": "category",
    "state_freq": "state",
}
for feature, source in frequency_sources.items():
    if feature not in feature_cols:
        continue
    previous_rows = m3_df.groupby(source, dropna=False).cumcount()
    same_timestamp_previous = m3_df.groupby(
        [source, TIME_COL], dropna=False
    ).cumcount()
    m3_df[feature] = (previous_rows - same_timestamp_previous).astype("int64")

train_mask = m3_df[SPLIT_COL].eq("train")
X_train_schema = m3_df.loc[train_mask, feature_cols]
categorical_cols = X_train_schema.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("Data shape:", m3_df.shape)
print("Features:", len(feature_cols))
print("Categorical features:", categorical_cols)
print("Fraud rates by split:")
display(m3_df.groupby(SPLIT_COL)[TARGET_COL].agg(["size", "mean"]))


Data shape: (1296675, 87)
Features: 62
Categorical features: ['category', 'gender', 'state']
Fraud rates by split:


,size,mean
sparkov_split,,
train,1037340,0.005753
validation,259335,0.005931


## Temporal setup

At each weekly cutoff, only transactions timestamped **strictly before `cutoff − 7 days`** are treated as verified training observations. Transactions in `[cutoff − 7 days, cutoff)` form the unverified candidate pool. A verified-only LightGBM model scores this pool; the rank-selected pseudo-fraud and pseudo-legitimate examples are added to the verified observations for a weighted refit. The refitted model scores transactions in the following seven-day block. This procedure repeats with expanding history.


In [2]:
# Fixed M2 temporal configuration
WINDOW_DAYS = None                 # expanding history
RETRAIN_EVERY_DAYS = 7
LABEL_DELAY_DAYS = 7
# Compact, prespecified 4-parameter grid (16 configurations).
# Lycia's selection rule and class-specific reliability weights are retained.
FRAUD_THRESHOLDS = [0.15, 0.50]
LEGIT_THRESHOLDS = [0.25, 0.50]
FRAUD_RELIABILITIES = [0.15, 0.50]
LEGIT_RELIABILITIES = [0.30, 0.75]


def make_lightgbm(class_weight="balanced"):
    """Best LightGBM tree settings from M1/M2."""
    return lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        objective="binary",
        class_weight=class_weight,
        is_unbalance=False,
        random_state=100,
        n_jobs=2,
        verbosity=-1,
    )


def prepare_features(frame, categories=None):
    X = frame[feature_cols].copy()
    for col in categorical_cols:
        if categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=categories[col])
    return X


def verified_training_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select verified labels available at this cutoff; use expanding history."""
    cutoff = pd.Timestamp(cutoff)
    latest_eligible = cutoff - pd.Timedelta(days=label_delay_days)
    mask = frame[SPLIT_COL].isin(allowed_splits) & frame[TIME_COL].lt(latest_eligible)
    return frame.loc[mask].copy()


def pseudo_candidate_rows(
    frame,
    cutoff,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
):
    """Select rows in the seven-day label-delay interval before the cutoff."""
    cutoff = pd.Timestamp(cutoff)
    start = cutoff - pd.Timedelta(days=label_delay_days)
    mask = (
        frame[SPLIT_COL].isin(allowed_splits)
        & frame[TIME_COL].ge(start)
        & frame[TIME_COL].lt(cutoff)
    )
    return frame.loc[mask].copy()


def make_cutoffs(start, end, frequency_days=RETRAIN_EVERY_DAYS):
    return pd.date_range(
        start=pd.Timestamp(start),
        end=pd.Timestamp(end),
        freq=f"{frequency_days}D",
        inclusive="left",
    )


def precompute_pseudo_scores(
    frame,
    start,
    end,
    allowed_splits=("train",),
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Cache one verified-only model's scores for each cutoff's delay window."""
    cache = {}
    cutoffs = make_cutoffs(start, end, frequency_days)

    for i, cutoff in enumerate(cutoffs, start=1):
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        candidates = pseudo_candidate_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        if verified.empty or verified[TARGET_COL].nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")
        if candidates.empty:
            cache[cutoff] = pd.DataFrame(
                columns=["row_index", "fraud_probability"]
            )
            continue

        X_verified = prepare_features(verified)
        categories = {col: X_verified[col].cat.categories for col in categorical_cols}
        X_candidates = prepare_features(candidates, categories=categories)
        model = make_lightgbm(class_weight="balanced")
        model.fit(X_verified, verified[TARGET_COL].astype(int))
        fraud_probability = model.predict_proba(X_candidates)[:, 1]

        cache[cutoff] = pd.DataFrame({
            "row_index": candidates.index.to_numpy(),
            "fraud_probability": fraud_probability,
        })
        print(
            f"Cached {i}/{len(cutoffs)} cutoffs: {cutoff} "
            f"({len(verified):,} verified; {len(candidates):,} unlabeled candidates)",
            flush=True,
        )
    return cache


def balanced_class_weight_map(y_verified):
    """Balanced class weights based only on verified labels at this cutoff."""
    classes = np.sort(pd.Series(y_verified).astype(int).unique())
    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=pd.Series(y_verified).astype(int).to_numpy(),
    )
    return {int(label): float(weight) for label, weight in zip(classes, weights)}


def summarize_predictions(predictions):
    y_true = predictions["y_true"].astype(int)
    y_score = predictions["y_score"].astype(float)
    return {
        "transactions": len(predictions),
        "fraud_cases": int(y_true.sum()),
        "pr_auc": average_precision_score(y_true, y_score),
        "roc_auc": roc_auc_score(y_true, y_score),
    }


In [5]:
def run_adaptive_lightgbm(
    frame,
    start,
    end,
    fraud_thres=None,
    legit_thres=None,
    fraud_reliability=0.0,
    legit_reliability=0.0,
    pseudo_score_cache=None,
    allowed_splits=("train",),
    scoring_split="train",
    label_delay_days=LABEL_DELAY_DAYS,
    frequency_days=RETRAIN_EVERY_DAYS,
):
    """Weekly LightGBM retraining; None fraud_thres gives verified-only M2."""
    start, end = pd.Timestamp(start), pd.Timestamp(end)
    if start >= end:
        raise ValueError("start must be earlier than end")
    use_pseudo = fraud_thres is not None
    if use_pseudo:
        if legit_thres is None:
            raise ValueError("legit_thres is required for M3")
        if not 0.0 < fraud_thres <= 1.5 or not 0.0 < legit_thres < 1.0:
            raise ValueError("fraud_thres must be in (0, 1.5], legit_thres in (0, 1)")
        if not 0 <= fraud_reliability <= 1 or not 0 <= legit_reliability <= 1:
            raise ValueError("Reliability weights must be in [0, 1]")

    prediction_blocks = []
    for cutoff in make_cutoffs(start, end, frequency_days):
        next_cutoff = min(cutoff + pd.Timedelta(days=frequency_days), end)
        verified = verified_training_rows(
            frame, cutoff, allowed_splits, label_delay_days
        )
        score_block = frame.loc[
            frame[SPLIT_COL].eq(scoring_split)
            & frame[TIME_COL].ge(cutoff)
            & frame[TIME_COL].lt(next_cutoff)
        ].copy()
        if score_block.empty:
            continue
        y_verified = verified[TARGET_COL].astype(int)
        if verified.empty or y_verified.nunique() < 2:
            raise ValueError(f"Insufficient verified classes at {cutoff}")

        pseudo_rows = frame.iloc[0:0].copy()
        pseudo_rows["_m3_target"] = pd.Series(dtype=int)
        pseudo_rows["_m3_reliability"] = pd.Series(dtype=float)
        if use_pseudo:
            if pseudo_score_cache is None or cutoff not in pseudo_score_cache:
                raise KeyError(f"Missing pseudo-label scores for {cutoff}")
            scored_candidates = pseudo_score_cache[cutoff]
            if not scored_candidates.empty:
                # Same rank-based top/bottom selection as Lycia's IEEE M3.
                p = scored_candidates["fraud_probability"].to_numpy()
                sorted_scores = np.sort(p)
                verified_rate = float(y_verified.mean())
                n_fraud = min(max(int(round(fraud_thres * verified_rate * len(p))), 1), len(p))
                n_legit = min(max(int(round(legit_thres * len(p))), 1), len(p))
                fraud_cutoff = sorted_scores[-n_fraud]
                legit_cutoff = sorted_scores[n_legit - 1]
                is_fraud = p >= fraud_cutoff
                is_legit = p <= legit_cutoff
                # Lycia's implementation gives fraud precedence when masks overlap.
                keep = is_fraud | is_legit
                chosen = scored_candidates.loc[keep]
                if not chosen.empty:
                    pseudo_rows = frame.loc[chosen["row_index"].astype(int).to_numpy()].copy()
                    pseudo_rows["_m3_target"] = is_fraud[keep].astype(int)
                    pseudo_rows["_m3_reliability"] = np.where(
                        is_fraud[keep], fraud_reliability, legit_reliability
                    )

        verified_rows = verified.copy()
        verified_rows["_m3_target"] = y_verified.to_numpy()
        fit_rows = pd.concat([verified_rows, pseudo_rows], axis=0)
        y_fit = fit_rows["_m3_target"].astype(int)

        # Per-row sample weight = class weight from verified labels × reliability.
        class_weights = balanced_class_weight_map(y_verified)
        class_component = y_fit.map(class_weights).to_numpy(dtype=float)
        reliability_component = np.concatenate([
            np.ones(len(verified_rows), dtype=float),
            pseudo_rows["_m3_reliability"].to_numpy(dtype=float),
        ])
        fit_sample_weight = class_component * reliability_component

        X_fit = prepare_features(fit_rows)
        categories = {col: X_fit[col].cat.categories for col in categorical_cols}
        X_score = prepare_features(score_block, categories=categories)
        # Class balance is already represented in fit_sample_weight, avoiding double weighting.
        model = make_lightgbm(class_weight=None)
        model.fit(X_fit, y_fit, sample_weight=fit_sample_weight)
        scores = model.predict_proba(X_score)[:, 1]

        prediction_blocks.append(pd.DataFrame({
            "row_index": score_block.index.to_numpy(),
            "timestamp": score_block[TIME_COL].to_numpy(),
            "y_true": score_block[TARGET_COL].astype(int).to_numpy(),
            "y_score": scores,
            "cutoff": cutoff,
            "fraud_thres": fraud_thres,
            "legit_thres": legit_thres,
            "fraud_reliability": fraud_reliability,
            "legit_reliability": legit_reliability,
            "verified_rows": len(verified_rows),
            "pseudo_rows": len(pseudo_rows),
            "class_weight_legitimate": class_weights[0],
            "class_weight_fraud": class_weights[1],
        }))

    if not prediction_blocks:
        raise ValueError("No scoring rows found")
    predictions = pd.concat(prediction_blocks, ignore_index=True)
    if predictions["row_index"].duplicated().any():
        raise AssertionError("Duplicate scored rows")
    return predictions


In [6]:

# Pre-run check: reproduce the final M2 LightGBM baseline

validation_rows_check = m3_df.loc[
    m3_df[SPLIT_COL].eq("validation")
]

val_start_check = validation_rows_check[TIME_COL].min()
val_end_check = (
    validation_rows_check[TIME_COL].max().floor("D")
    + pd.Timedelta(days=1)
)

baseline_check = run_adaptive_lightgbm(
    frame=m3_df,
    start=val_start_check,
    end=val_end_check,
    fraud_thres=None,
    allowed_splits=("train", "validation"),
    scoring_split="validation",
)

baseline_pr_auc = summarize_predictions(
    baseline_check
)["pr_auc"]

print(f"Notebook 5 M2 PR-AUC: 0.985243")
print(f"Notebook 6 M2 PR-AUC: {baseline_pr_auc:.6f}")

assert np.isclose(
    baseline_pr_auc,
    0.985243,
    atol=1e-5,
    rtol=0
), "M2 baseline mismatch — investigate before running M3."

print("M2 baseline reproduced successfully.")


Notebook 5 M2 PR-AUC: 0.985243
Notebook 6 M2 PR-AUC: 0.985243
M2 baseline reproduced successfully.


## Development tuning

Use the 16-configuration grid defined above to tune `fraud_thres`, `legit_thres`, `fraud_reliability`, and `legit_reliability` on the final 84 days of the **training** partition. Rank-based pseudo-label selection and separate reliability weights follow IEEE-CIS M3; the classifier, class balancing, and compact grid remain Sparkov-specific. Select the configuration with the highest development PR-AUC. Cached verified-only candidate scores are reused across configurations to reduce repeated fitting.


In [7]:
# Development window: the final 84 days of the original training period.
train_end = m3_df.loc[
    m3_df[SPLIT_COL].eq("train"), TIME_COL
].max()

DEV_END = train_end.floor("D") + pd.Timedelta(days=1)
DEV_START = DEV_END - pd.Timedelta(days=84)
print("Development start:", DEV_START)
print("Development end:  ", DEV_END)

DEV_PSEUDO_CACHE = precompute_pseudo_scores(
    frame=m3_df,
    start=DEV_START,
    end=DEV_END,
    allowed_splits=("train",),
)


def config_id(fraud_thres, legit_thres, fraud_reliability, legit_reliability):
    return (f"f{fraud_thres:.2f}_l{legit_thres:.2f}_"
            f"fw{fraud_reliability:.2f}_lw{legit_reliability:.2f}")


def tune_m3(frame, overwrite=False):
    results_path = RESULTS_DIR / "m3_development_grid.csv"
    rows = []
    configs = list(product(FRAUD_THRESHOLDS, LEGIT_THRESHOLDS,
                           FRAUD_RELIABILITIES, LEGIT_RELIABILITIES))
    reference_rows = None

    for i, (ft, lt, fr, lr) in enumerate(configs, start=1):
        name = config_id(ft, lt, fr, lr)
        prediction_path = RESULTS_DIR / f"development_{name}.csv"
        print(f"\n[{i}/{len(configs)}] {name}", flush=True)
        started = time.perf_counter()

        if prediction_path.exists() and not overwrite:
            pred = pd.read_csv(prediction_path, parse_dates=["timestamp", "cutoff"])
        else:
            pred = run_adaptive_lightgbm(
                frame=frame, start=DEV_START, end=DEV_END,
                fraud_thres=ft, legit_thres=lt,
                fraud_reliability=fr, legit_reliability=lr,
                pseudo_score_cache=DEV_PSEUDO_CACHE,
                allowed_splits=("train",), scoring_split="train",
            )
            pred.to_csv(prediction_path, index=False)

        metrics = summarize_predictions(pred)
        row = {"config_id": name, "fraud_thres": ft, "legit_thres": lt,
               "fraud_reliability": fr, "legit_reliability": lr,
               **metrics, "mean_pseudo_rows_per_retrain": pred["pseudo_rows"].mean()}
        rows.append(row)
        pd.DataFrame(rows).to_csv(results_path, index=False)

        scored_rows = set(pred["row_index"].astype(int))
        if reference_rows is None:
            reference_rows = scored_rows
        else:
            assert scored_rows == reference_rows, f"Scored rows differ for {name}"

        print(f"PR-AUC={metrics['pr_auc']:.4f} | ROC-AUC={metrics['roc_auc']:.4f} | "
              f"Mean pseudo rows/retrain={row['mean_pseudo_rows_per_retrain']:.0f} | "
              f"Elapsed={time.perf_counter() - started:.1f}s", flush=True)

    return pd.DataFrame(rows).sort_values(
        ["pr_auc", "roc_auc"], ascending=[False, False]
    ).reset_index(drop=True)


development_results = tune_m3(m3_df)
display(development_results)
best_config = development_results.iloc[0].to_dict()
print("Selected configuration:")
display(pd.DataFrame([best_config]))
pd.DataFrame([best_config]).to_csv(
    RESULTS_DIR / "m3_best_config.csv", index=False
)


Development start: 2019-12-14 00:00:00
Development end:   2020-03-07 00:00:00
Cached 1/12 cutoffs: 2019-12-14 00:00:00 (809,930 verified; 31,595 unlabeled candidates)
Cached 2/12 cutoffs: 2019-12-21 00:00:00 (841,525 verified; 31,393 unlabeled candidates)
Cached 3/12 cutoffs: 2019-12-28 00:00:00 (872,918 verified; 31,100 unlabeled candidates)
Cached 4/12 cutoffs: 2020-01-04 00:00:00 (904,018 verified; 24,885 unlabeled candidates)
Cached 5/12 cutoffs: 2020-01-11 00:00:00 (928,903 verified; 11,974 unlabeled candidates)
Cached 6/12 cutoffs: 2020-01-18 00:00:00 (940,877 verified; 12,199 unlabeled candidates)
Cached 7/12 cutoffs: 2020-01-25 00:00:00 (953,076 verified; 12,030 unlabeled candidates)
Cached 8/12 cutoffs: 2020-02-01 00:00:00 (965,106 verified; 11,946 unlabeled candidates)
Cached 9/12 cutoffs: 2020-02-08 00:00:00 (977,052 verified; 11,949 unlabeled candidates)
Cached 10/12 cutoffs: 2020-02-15 00:00:00 (989,001 verified; 11,797 unlabeled candidates)
Cached 11/12 cutoffs: 2020-02-2

,config_id,fraud_thres,legit_thres,fraud_reliability,legit_reliability,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,f0.50_l0.50_fw0.50_lw0.30,0.50,0.50,0.50,0.30,195815,1101,0.986481,0.999802,10778.208779
1,f0.50_l0.25_fw0.50_lw0.75,0.50,0.25,0.50,0.75,195815,1101,0.986427,0.999785,5419.984833
2,f0.15_l0.50_fw0.50_lw0.30,0.15,0.50,0.50,0.30,195815,1101,0.986376,0.999759,10734.863361
3,f0.50_l0.25_fw0.15_lw0.30,0.50,0.25,0.15,0.30,195815,1101,0.986304,0.999795,5419.984833
4,f0.50_l0.25_fw0.50_lw0.30,0.50,0.25,0.50,0.30,195815,1101,0.986239,0.999791,5419.984833
5,f0.50_l0.50_fw0.50_lw0.75,0.50,0.50,0.50,0.75,195815,1101,0.986200,0.999769,10778.208779
6,f0.15_l0.25_fw0.50_lw0.75,0.15,0.25,0.50,0.75,195815,1101,0.986162,0.999795,5376.639415
7,f0.50_l0.50_fw0.15_lw0.30,0.50,0.50,0.15,0.30,195815,1101,0.985925,0.999758,10778.208779
8,f0.15_l0.50_fw0.15_lw0.30,0.15,0.50,0.15,0.30,195815,1101,0.985908,0.999793,10734.863361
9,f0.50_l0.25_fw0.15_lw0.75,0.50,0.25,0.15,0.75,195815,1101,0.985796,0.999775,5419.984833


Selected configuration:


,config_id,fraud_thres,legit_thres,fraud_reliability,legit_reliability,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,f0.50_l0.50_fw0.50_lw0.30,0.5,0.5,0.5,0.3,195815,1101,0.986481,0.999802,10778.208779


## Held-out validation

Using the development-selected four-parameter configuration, compare M3 with the **verified-only M2 LightGBM baseline** on the same validation transactions and weekly retraining schedule. The baseline uses no pseudo-labels; M3 uses rank-selected pseudo-labels with class-specific reliability multipliers. Compare validation PR-AUC and ROC-AUC; do not tune the M3 parameters on validation results. The test partition is not used here.


In [8]:
validation_rows = m3_df.loc[m3_df[SPLIT_COL].eq("validation")]
VAL_START = validation_rows[TIME_COL].min()
VAL_END = (
    validation_rows[TIME_COL].max().floor("D")
    + pd.Timedelta(days=1)
)
print("Validation rows:", len(validation_rows))
print("Validation start:", VAL_START)
print("Validation end:  ", VAL_END)

VAL_PSEUDO_CACHE = precompute_pseudo_scores(
    frame=m3_df,
    start=VAL_START,
    end=VAL_END,
    allowed_splits=("train", "validation"),
)

# M2 reference: same temporal setup, but no pseudo-labeled observations.
m2_baseline_predictions = run_adaptive_lightgbm(
    frame=m3_df,
    start=VAL_START,
    end=VAL_END,
    fraud_thres=None,
    pseudo_score_cache=None,
    allowed_splits=("train", "validation"),
    scoring_split="validation",
)

m3_validation_predictions = run_adaptive_lightgbm(
    frame=m3_df,
    start=VAL_START,
    end=VAL_END,
    fraud_thres=float(best_config["fraud_thres"]),
    legit_thres=float(best_config["legit_thres"]),
    fraud_reliability=float(best_config["fraud_reliability"]),
    legit_reliability=float(best_config["legit_reliability"]),
    pseudo_score_cache=VAL_PSEUDO_CACHE,
    allowed_splits=("train", "validation"),
    scoring_split="validation",
)

expected_rows = set(m3_df.index[m3_df[SPLIT_COL].eq("validation")])
for label, pred in [
    ("M2 baseline", m2_baseline_predictions),
    ("M3 pseudo-labeling", m3_validation_predictions),
]:
    assert len(pred) == len(expected_rows)
    assert set(pred["row_index"].astype(int)) == expected_rows

m2_metrics = summarize_predictions(m2_baseline_predictions)
m3_metrics = summarize_predictions(m3_validation_predictions)


# Verify that the M3 notebook reproduces the completed M2 baseline.
EXPECTED_M2_PR_AUC = 0.985243

print(
    f"Original M2 PR-AUC: {EXPECTED_M2_PR_AUC:.6f}"
)
print(
    f"M3 notebook M2 PR-AUC: {m2_metrics['pr_auc']:.6f}"
)

assert np.isclose(
    m2_metrics["pr_auc"],
    EXPECTED_M2_PR_AUC,
    atol=1e-5,
    rtol=0
), (
    "M2 baseline does not match Notebook 5. "
    "Investigate before interpreting M3 improvements."
)

print("M2 baseline alignment verified.")


validation_comparison = pd.DataFrame([
    {
        "model": "LightGBM",
        "strategy": "M2 balanced, verified labels only",
        "fraud_thres": np.nan,
        "legit_thres": np.nan,
        "fraud_reliability": np.nan,
        "legit_reliability": np.nan,
        **m2_metrics,
    },
    {
        "model": "LightGBM",
        "strategy": "M3 balanced, pseudo-labeling",
        "fraud_thres": float(best_config["fraud_thres"]),
        "legit_thres": float(best_config["legit_thres"]),
        "fraud_reliability": float(best_config["fraud_reliability"]),
        "legit_reliability": float(best_config["legit_reliability"]),
        **m3_metrics,
        "mean_pseudo_rows_per_retrain": m3_validation_predictions["pseudo_rows"].mean(),
    },
])
display(validation_comparison)
validation_comparison.to_csv(RESULTS_DIR / "m3_validation_comparison.csv", index=False)
m3_validation_predictions.to_csv(
    RESULTS_DIR / "m3_validation_predictions.csv", index=False
)


Validation rows: 259335
Validation start: 2020-03-06 07:16:43
Validation end:   2020-06-22 00:00:00
Cached 1/16 cutoffs: 2020-03-06 07:16:43 (1,023,767 verified; 13,573 unlabeled candidates)
Cached 2/16 cutoffs: 2020-03-13 07:16:43 (1,037,340 verified; 16,114 unlabeled candidates)
Cached 3/16 cutoffs: 2020-03-20 07:16:43 (1,053,454 verified; 16,087 unlabeled candidates)
Cached 4/16 cutoffs: 2020-03-27 07:16:43 (1,069,541 verified; 15,950 unlabeled candidates)
Cached 5/16 cutoffs: 2020-04-03 07:16:43 (1,085,491 verified; 15,855 unlabeled candidates)
Cached 6/16 cutoffs: 2020-04-10 07:16:43 (1,101,346 verified; 16,057 unlabeled candidates)
Cached 7/16 cutoffs: 2020-04-17 07:16:43 (1,117,403 verified; 15,984 unlabeled candidates)
Cached 8/16 cutoffs: 2020-04-24 07:16:43 (1,133,387 verified; 15,766 unlabeled candidates)
Cached 9/16 cutoffs: 2020-05-01 07:16:43 (1,149,153 verified; 15,872 unlabeled candidates)
Cached 10/16 cutoffs: 2020-05-08 07:16:43 (1,165,025 verified; 15,677 unlabeled c

,model,strategy,fraud_thres,legit_thres,fraud_reliability,legit_reliability,transactions,fraud_cases,pr_auc,roc_auc,mean_pseudo_rows_per_retrain
0,LightGBM,"M2 balanced, verified labels only",NaN,NaN,NaN,NaN,259335,1538,0.985243,0.999628,NaN
1,LightGBM,"M3 balanced, pseudo-labeling",0.5,0.5,0.5,0.3,259335,1538,0.985322,0.999523,8400.579266



## M3 Results and Interpretation

### Development Tuning

We evaluated 16 configurations of confidence-based pseudo-labelling using a chronological 84-day development window. Following the IEEE-CIS M3 methodology, pseudo-fraud and pseudo-legitimate transactions were selected separately based on their predicted fraud-risk rankings, with class-specific reliability weights applied during retraining.

The best configuration achieved a development PR-AUC of **0.986481** and ROC-AUC of **0.999802**, using:

| Parameter | Selected Value |
|---|---:|
| `fraud_thres` | 0.50 |
| `legit_thres` | 0.50 |
| `fraud_reliability` | 0.50 |
| `legit_reliability` | 0.30 |

The selected configuration pseudo-labelled the highest-risk transactions corresponding to 0.50 times the observed verified fraud prevalence as fraudulent, and the lowest-risk 50% as legitimate. Pseudo-labelled observations received reduced reliability weights, while the existing M2 class-balancing strategy was retained.

### Held-Out Validation Results

The selected M3 configuration was evaluated against the verified-only M2 LightGBM baseline on the same chronological validation set, comprising **259,335 transactions and 1,538 fraudulent transactions**.

| Metric | M2 LightGBM | M3 LightGBM |
|---|---:|---:|
| PR-AUC | 0.985243 | **0.985322** |
| ROC-AUC | **0.999628** | 0.999523 |

M3 achieved a marginal PR-AUC improvement of **0.000079** over M2, while ROC-AUC decreased by 0.000105. This indicates a marginal improvement in validation PR-AUC, although the difference is too small to establish a meaningful performance advantage without further robustness testing.

### Interpretation

The results suggest that semi-supervised pseudo-labelling offers limited incremental benefit on the Sparkov dataset, where the M2 LightGBM baseline already achieves exceptionally high discrimination performance.

Although M3 slightly improved validation PR-AUC, the magnitude of improvement is insufficient to establish a meaningful or robust performance advantage without additional testing. Furthermore, the decrease in ROC-AUC indicates that the improvement is not consistent across both ranking metrics.

Therefore, M3 demonstrates the feasibility of incorporating recent unverified transactions under a seven-day label verification delay, but does not provide compelling evidence that the additional computational complexity is justified by predictive improvements.

The M3 findings will inform subsequent M4 evaluation and final model selection, with PR-AUC remaining the primary model-selection metric.

